In [1]:
# 1. Kiểm tra môi trường GPU CUDA và cấu hình máy ảo
import os
import torch
import sys

print("======== ENVIRONMENT DIAGNOSTICS ========")
print(f"🐍 Python Version: {sys.version}")
print(f"🔥 PyTorch Version: {torch.__version__}")
print(f"🔥 CUDA Available: {torch.cuda.is_available()}")
print(f"🔥 GPU Device Count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"   └── GPU [{i}]: {torch.cuda.get_device_name(i)}")
else:
    print("⚠️ CẢNH BÁO: Không tìm thấy GPU CUDA! Vui lòng bật accelerator GPU T4x2 trong cài đặt Kaggle Notebook.")
print("=========================================")

In [ ]:
# 2. Cài đặt các thư viện cần thi💻 và gỡ cài đặt torchao bị xung đột trên Kaggle
!pip uninstall -y torchao
!pip install -q transformers datasets peft bitsandbytes accelerate jiwer soundfile librosa pandas click matplotlib tabulate

import transformers
import peft
import datasets
import accelerate
import bitsandbytes

print("\n======== INSTALLED PACKAGES ========")
print(f"📦 transformers: {transformers.__version__}")
print(f"📦 peft: {peft.__version__}")
print(f"📦 datasets: {datasets.__version__}")
print(f"📦 accelerate: {accelerate.__version__}")
print(f"📦 bitsandbytes: {bitsandbytes.__version__}")
print("====================================")

In [ ]:
# 3. Clone mã nguồn từ GitHub (nhánh benchmark/scaling-laws-results)
import os

# Để clone từ private repo, bạn có 2 cách:
# Cách 1 (Khuyên dùng): Vào Add-ons -> Secrets trên Kaggle, tạo một Secret tên "GITHUB_PAT" chứa GitHub Personal Access Token.
# Cách 2: Điền trực tiếp Token vào biến github_pat dưới đây.
github_pat = ""

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    github_pat = user_secrets.get_secret("GITHUB_PAT")
except Exception:
    pass

if github_pat:
    clone_url = f"https://{github_pat}@github.com/silvermango9927/synthetic-data-pipeline.git"
else:
    clone_url = "https://github.com/silvermango9927/synthetic-data-pipeline.git"

if not os.path.exists("synthetic-data-pipeline"):
    !git clone -b benchmark/scaling-laws-results {clone_url}
else:
    print("Codebase đã tồn tại. Tiến hành git pull để cập nhật...")
    %cd synthetic-data-pipeline
    !git remote set-url origin {clone_url}
    !git pull origin benchmark/scaling-laws-results
    %cd ..

In [ ]:
# 4. Tạo symbolic link để trỏ dataset từ Kaggle input vào đúng thư mục code mong đợi
import os
from pathlib import Path

# Thư mục gốc chứa dataset trên Kaggle (tiếng Hindi)
kaggle_dataset_path = "/kaggle/input/datasets/trihuynhviprovcl/synthetic-asr-hi/synthetic-asr-hi"
# Thư mục đích trong workspace code
local_target_path = "/kaggle/working/synthetic-data-pipeline/outputs/hf_datasets/synthetic-asr-hi"

# Tạo thư mục cha nếu chưa có
os.makedirs(os.path.dirname(local_target_path), exist_ok=True)

# Tạo symbolic link
if os.path.exists(local_target_path):
    if os.path.islink(local_target_path):
        os.unlink(local_target_path)
    else:
        import shutil
        shutil.rmtree(local_target_path)

os.symlink(kaggle_dataset_path, local_target_path)
print(f"🔗 Đã liên kết dataset từ Kaggle sang: {local_target_path}")

# Kiểm tra thử đường dẫn tẹp tin manifest để đảm bảo kết nối hoạt động tốt
test_file = Path(local_target_path) / "data" / "long_clean" / "manifest.jsonl"
if test_file.exists():
    print("✅ Kết nối dữ liệu thành công!")
    with open(test_file, "r", encoding="utf-8") as f:
        samples = sum(1 for _ in f)
    print(f"📊 Số lượng mẫu âm thanh được tìm thấy: {samples} mẫu.")
else:
    print("❌ Lỗi: Không tìm thấy tẹp tin dữ liệu. Vui lòng kiểm tra lại cấu trúc dataset Kaggle input.")

In [ ]:
# 5. Chạy ASR Scaling Sweep và ghi nhật ký (Log) ra tẹp tin
%cd /kaggle/working/synthetic-data-pipeline

# Đảm bảo thư mục lưu log tồn tại
os.makedirs("outputs/benchmark/stats", exist_ok=True)

# Cấu hình mặc định fp16 + LoRA để tốc đ🔥 nhanh nhất, không bị OOM trên 1 GPU T4 (15GB VRAM).
# Toàn bộ log được hiển th📺 và ghi vào training.log
!CUDA_VISIBLE_DEVICES=0 python -m benchmark.scaling \
  --model-name openai/whisper-large-v3-turbo \
  --use-lora \
  --epochs 3 \
  --batch-size 4 \
  --lr 5e-5 \
  --fractions "0.1,0.2,0.4,0.6,0.8,1.0" \
  --lang hi 2>&1 | tee outputs/benchmark/stats/training.log

# CẦU HÌNH TEST NHANH PIPELINE (Hãy uncomment dòng lệnh dưới nếu muốn test thử trước):
# !CUDA_VISIBLE_DEVICES=0 python -m benchmark.scaling \
#   --model-name openai/whisper-tiny \
#   --use-lora \
#   --epochs 1 \
#   --batch-size 16 \
#   --lr 1e-4 \
#   --fractions "0.1" \
#   --lang hi 2>&1 | tee outputs/benchmark/stats/training.log

In [ ]:
# 6. Copy kết quả ra ngoài và hiển thị biểu đồ kết quả Scaling Laws
from IPython.display import Image, display
import shutil
import glob
import os

# Copy toàn bộ kết quả và tẹp log ra thư mục gốc /kaggle/working để Kaggle Output lưu lại độc lập
os.makedirs("/kaggle/working/stats", exist_ok=True)
for f in glob.glob("outputs/benchmark/stats/*"):
    shutil.copy(f, "/kaggle/working/stats/")
print("✅ Đã copy toàn bộ kết quả huấn luyện và tẹp nhật ký sang thư mục lưu trữ: /kaggle/working/stats/")

# Tìm file biểu đồ .png được sinh ra
plot_files = glob.glob("/kaggle/working/stats/scaling_*.png")
if plot_files:
    print(f"📈 Hiển thị biểu đồ: {plot_files[0]}")
    display(Image(filename=plot_files[0]))
else:
    print("❌ Không tìm thấy biểu đồ kết quả. Vui lòng kiểm tra xem sweep đã chạy xong chưa.")